# Gikomba Stall Performance Analysis

**Analyst:** Enoch Baraka  
**Client:** Wanjiku, fresh-produce stall at Gikomba Market  
**Question:** How is my business doing? Is it growing? What should I restock, and where should I focus?

This notebook uses **core Python only** (no pandas or NumPy) to analyse 14 days of sales and the current stock record.

## Dataset

The data below is copied exactly as provided: 14 days of daily sales (KES), the day names, and the stock record.

In [1]:
daily_sales = [2300, 1950, 2800, 2100, 3600, 5200, 4100,
               2500, 2200, 3100, 2400, 3900, 5600, 4700]   # 14 days, KES

days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"] * 2

# qty = kg in stock, reorder = reorder level (kg), cost/price = KES per kg
stock = {
    "Tomatoes": {"qty": 18, "reorder": 25, "cost": 60, "price": 90},
    "Onions":   {"qty": 40, "reorder": 20, "cost": 70, "price": 100},
    "Sukuma":   {"qty": 9,  "reorder": 15, "cost": 30, "price": 50},
    "Potatoes": {"qty": 55, "reorder": 30, "cost": 50, "price": 75},
    "Avocado":  {"qty": 12, "reorder": 10, "cost": 80, "price": 140},
}

## Task 1: Summary statistics and best/worst days

**Approach:** I use `sum()`, `len()`, `max()` and `min()` for the totals and extremes. Then `list.index()` gives each extreme's position in the list. The day name comes from `days[position]`, and the week number is `position // 7 + 1`, because positions 0 to 6 are week 1 and 7 to 13 are week 2.

In [2]:
total_sales = sum(daily_sales)
average_sales = total_sales / len(daily_sales)
highest = max(daily_sales)
lowest = min(daily_sales)

best_pos = daily_sales.index(highest)
worst_pos = daily_sales.index(lowest)

best_day, best_week = days[best_pos], best_pos // 7 + 1
worst_day, worst_week = days[worst_pos], worst_pos // 7 + 1

print(f"Total sales:   KES {total_sales:,}")
print(f"Average daily: KES {average_sales:,.2f}")
print(f"Highest day:   KES {highest:,} ({best_day}, week {best_week})")
print(f"Lowest day:    KES {lowest:,} ({worst_day}, week {worst_week})")

Total sales:   KES 46,450
Average daily: KES 3,317.86
Highest day:   KES 5,600 (Sat, week 2)
Lowest day:    KES 1,950 (Tue, week 1)


## Task 2: Weekly totals and week-on-week growth

**Approach:** `week_totals()` slices the list into `sales[:7]` (week 1) and `sales[7:14]` (week 2) and returns both totals as a tuple. I unpack the tuple and compute growth as (week 2 − week 1) ÷ week 1 × 100.

In [3]:
def week_totals(sales):
    """Return a tuple (week 1 total, week 2 total) from a list of 14 daily sales.

    Week 1 is the first seven values (sales[:7]) and week 2 the next seven (sales[7:14]).
    """
    return sum(sales[:7]), sum(sales[7:14])


week1_total, week2_total = week_totals(daily_sales)
growth_pct = (week2_total - week1_total) / week1_total * 100

print(f"Week 1 total: KES {week1_total:,}")
print(f"Week 2 total: KES {week2_total:,}")
print(f"Week-on-week growth: {growth_pct:.1f}%")

Week 1 total: KES 22,050
Week 2 total: KES 24,400
Week-on-week growth: 10.7%


## Task 3: Classifying each day

**Approach:** `classify_day()` compares a day's sales to the average. Above 120% of the average is "Strong", below 80% is "Weak", and anything else is "Normal". I apply it to every day and count the categories with the dictionary counting pattern (`counts[label] = counts.get(label, 0) + 1`).

In [4]:
def classify_day(amount, average):
    """Classify a day's sales relative to the average.

    Returns "Strong" if amount is more than 20% above average,
    "Weak" if it is more than 20% below average, otherwise "Normal".
    """
    if amount > average * 1.2:
        return "Strong"
    elif amount < average * 0.8:
        return "Weak"
    return "Normal"


day_labels = []
counts = {}
for i, amount in enumerate(daily_sales):
    label = classify_day(amount, average_sales)
    day_labels.append(label)
    counts[label] = counts.get(label, 0) + 1
    week = i // 7 + 1
    print(f"Week {week} {days[i]}: KES {amount:>5,}  ->  {label}")

print()
print("Day counts:", counts)

Week 1 Mon: KES 2,300  ->  Weak
Week 1 Tue: KES 1,950  ->  Weak
Week 1 Wed: KES 2,800  ->  Normal
Week 1 Thu: KES 2,100  ->  Weak
Week 1 Fri: KES 3,600  ->  Normal
Week 1 Sat: KES 5,200  ->  Strong
Week 1 Sun: KES 4,100  ->  Strong
Week 2 Mon: KES 2,500  ->  Weak
Week 2 Tue: KES 2,200  ->  Weak
Week 2 Wed: KES 3,100  ->  Normal
Week 2 Thu: KES 2,400  ->  Weak
Week 2 Fri: KES 3,900  ->  Normal
Week 2 Sat: KES 5,600  ->  Strong
Week 2 Sun: KES 4,700  ->  Strong

Day counts: {'Weak': 6, 'Normal': 4, 'Strong': 4}


## Task 4: Weekdays versus weekends

**Approach:** Position in the week is `i % 7`. Positions 0 to 4 are Mon to Fri and positions 5 and 6 are Sat and Sun. I collect the weekday and weekend sales into two lists, average each, and express the gap as a percentage of the weekday average.

In [5]:
weekday_sales = []
weekend_sales = []
for i, amount in enumerate(daily_sales):
    if i % 7 <= 4:
        weekday_sales.append(amount)
    else:
        weekend_sales.append(amount)

weekday_avg = sum(weekday_sales) / len(weekday_sales)
weekend_avg = sum(weekend_sales) / len(weekend_sales)
weekend_premium_pct = (weekend_avg - weekday_avg) / weekday_avg * 100

print(f"Weekday average (Mon-Fri): KES {weekday_avg:,.2f}")
print(f"Weekend average (Sat-Sun): KES {weekend_avg:,.2f}")
print(f"Weekend sales are {weekend_premium_pct:.1f}% higher than weekday sales")

Weekday average (Mon-Fri): KES 2,685.00
Weekend average (Sat-Sun): KES 4,900.00
Weekend sales are 82.5% higher than weekday sales


## Task 5: Reorder list and gross margins

**Approach:** `products_to_reorder()` loops over `stock.items()` and collects every product whose quantity is at or below its reorder level. Gross margin % is (price − cost) ÷ price × 100, calculated for each product in a second loop.

In [6]:
def products_to_reorder(stock_dict):
    """Return a list of product names whose qty is at or below their reorder level."""
    low = []
    for name, info in stock_dict.items():
        if info["qty"] <= info["reorder"]:
            low.append(name)
    return low


reorder_list = products_to_reorder(stock)

margins = {}
for name, info in stock.items():
    margins[name] = (info["price"] - info["cost"]) / info["price"] * 100

print("Products to reorder:", ", ".join(reorder_list))
print()
print("Gross margin by product:")
for name, margin in margins.items():
    print(f"  {name:<9} {margin:.1f}%")

best_margin_product = max(margins, key=margins.get)
print(f"\nHighest margin: {best_margin_product} ({margins[best_margin_product]:.1f}%)")

Products to reorder: Tomatoes, Sukuma

Gross margin by product:
  Tomatoes  33.3%
  Onions    30.0%
  Sukuma    40.0%
  Potatoes  33.3%
  Avocado   42.9%

Highest margin: Avocado (42.9%)


## Task 6: Stall Performance Report

**Approach:** All the earlier results are brought together in one report printed with f-strings. Money uses the `:,` format specifier so amounts show as `KES 46,450`.

In [7]:
line = "=" * 46
print(line)
print("        STALL PERFORMANCE REPORT - GIKOMBA")
print(line)
print("SALES (14 days)")
print(f"  Total sales:        KES {total_sales:>8,}")
print(f"  Average per day:    KES {average_sales:>8,.2f}")
print(f"  Best day:           KES {highest:>8,} ({best_day}, week {best_week})")
print(f"  Worst day:          KES {lowest:>8,} ({worst_day}, week {worst_week})")
print()
print("GROWTH")
print(f"  Week 1 total:       KES {week1_total:>8,}")
print(f"  Week 2 total:       KES {week2_total:>8,}")
print(f"  Week-on-week:       {growth_pct:>+8.1f}%")
print()
print("DAY QUALITY")
for label in ("Strong", "Normal", "Weak"):
    print(f"  {label:<8} days:      {counts.get(label, 0):>8}")
print()
print("WEEKDAY VS WEEKEND")
print(f"  Weekday average:    KES {weekday_avg:>8,.2f}")
print(f"  Weekend average:    KES {weekend_avg:>8,.2f}")
print(f"  Weekend premium:    {weekend_premium_pct:>+8.1f}%")
print()
print("STOCK")
print(f"  Reorder now:        {', '.join(reorder_list)}")
print("  Gross margins:")
for name, margin in margins.items():
    print(f"    {name:<9} {margin:>5.1f}%")
print(line)

        STALL PERFORMANCE REPORT - GIKOMBA
SALES (14 days)
  Total sales:        KES   46,450
  Average per day:    KES 3,317.86
  Best day:           KES    5,600 (Sat, week 2)
  Worst day:          KES    1,950 (Tue, week 1)

GROWTH
  Week 1 total:       KES   22,050
  Week 2 total:       KES   24,400
  Week-on-week:          +10.7%

DAY QUALITY
  Strong   days:             4
  Normal   days:             4
  Weak     days:             6

WEEKDAY VS WEEKEND
  Weekday average:    KES 2,685.00
  Weekend average:    KES 4,900.00
  Weekend premium:       +82.5%

STOCK
  Reorder now:        Tomatoes, Sukuma
  Gross margins:
    Tomatoes   33.3%
    Onions     30.0%
    Sukuma     40.0%
    Potatoes   33.3%
    Avocado    42.9%


## Recommendations for Wanjiku

1. **Stock up before the weekend:** Saturday and Sunday average KES 4,900 a day compared with KES 2,685 on weekdays (82.5% more), so make sure shelves are full by Friday evening.
2. **Reorder Sukuma and Tomatoes now:** both are below their reorder levels (9 kg against 15 kg, and 18 kg against 25 kg), and a stock-out on a weekend would cost you your best days.
3. **Push Avocado:** it has the highest margin at 42.9%, ahead of Onions at 30.0% and Tomatoes at 33.3%, so a little more shelf space and a promotion could lift profit more than extra volume of lower-margin items.
4. **Keep growing:** sales rose 10.7% from KES 22,050 in week 1 to KES 24,400 in week 2, so keep doing what worked and track a few more weeks to confirm the trend.
5. **Work on the quiet days:** 6 of the 14 days were "Weak" (more than 20% below the KES 3,317.86 average), so try a small midweek offer to lift Tuesday and Thursday sales.